# Lab 6 — Lemmatization
*NLP · answers notebook*

In [1]:
import nltk
# nltk.download("wordnet"); nltk.download("omw-1.4")
# nltk.download("punkt_tab"); nltk.download("averaged_perceptron_tagger_eng")
from nltk.stem import WordNetLemmatizer
from nltk import pos_tag, word_tokenize

## Exercise 1 — Basic lemmatization and its default assumption

In [2]:
lemmatizer = WordNetLemmatizer()

words = ["dogs", "studies", "running", "better", "geese", "was"]

for word in words:
    print(word, "->", lemmatizer.lemmatize(word))

dogs -> dog
studies -> study
running -> running
better -> better
geese -> goose
was -> wa


**1.1 Outputs:** `dogs→dog`, `studies→study`, `running→running`, `better→better`, `geese→goose`, `was→wa`.

**1.2 Wrongly reduced words:** `running` (expected *run*), `better` (expected *good*) and `was` (expected *be*; it even comes out as the non-word `wa`, because the lemmatizer strips the plural-looking `-s` as if *was* were a noun). The lemmatizer assumes that **every input word is a noun**. Nouns like *dogs*, *studies* and *geese* work; verbs and adjectives don't.

**1.3 Default `pos`:** `pos="n"`, which stands for **noun**. (Other values: `"v"` verb, `"a"` adjective, `"r"` adverb, `"s"` satellite adjective.)

**1.4 Prediction:** `lemmatize("geese", pos="n")` should return **`goose`**, because *geese* is in WordNet's irregular-noun exception list.

In [3]:
print(lemmatizer.lemmatize("geese", pos="n"))

goose


Verified: `goose`. (It already gave `goose` in the loop above, since `n` is the default.)

## Exercise 2 — Lemmatization with explicit POS

In [4]:
print(lemmatizer.lemmatize("running", pos="v"))
print(lemmatizer.lemmatize("better", pos="a"))
print(lemmatizer.lemmatize("was", pos="v"))
print(lemmatizer.lemmatize("geese", pos="n"))

run
good
be
goose


**2.1 Comparison with Exercise 1:** `running` → **run** (was `running`), `better` → **good** (was `better`), `was` → **be** (was `wa`), `geese` → `goose` (unchanged). Giving the right POS fixes the three words that were wrong.

**2.2 `POS_MAP`** (Penn Treebank tags from `pos_tag` → WordNet codes):

In [5]:
POS_MAP = {
    # nouns
    "NN": "n", "NNS": "n", "NNP": "n", "NNPS": "n",
    # verbs
    "VB": "v", "VBD": "v", "VBG": "v", "VBN": "v", "VBP": "v", "VBZ": "v",
    # adjectives
    "JJ": "a", "JJR": "a", "JJS": "a",
    # adverbs
    "RB": "r", "RBR": "r", "RBS": "r",
}

def pos_aware_lemmatize(sentence):
    """Tokenize, POS-tag, map tags with POS_MAP (default 'n') and lemmatize each token."""
    tagged = pos_tag(word_tokenize(sentence))
    return [(tok, tag, lemmatizer.lemmatize(tok.lower(), POS_MAP.get(tag, "n")))
            for tok, tag in tagged]

**2.4 Test sentence:** naive (no POS) vs POS-aware, word by word.

In [6]:
sentence = "The striped bats are hanging on their feet for best results."

import pandas as pd
aware = pos_aware_lemmatize(sentence)
cmp = pd.DataFrame({
    "token":  [t for t, _, _ in aware],
    "tag":    [g for _, g, _ in aware],
    "naive":  [lemmatizer.lemmatize(t.lower()) for t, _, _ in aware],
    "aware":  [l for _, _, l in aware],
})
cmp["differs"] = cmp["naive"] != cmp["aware"]
cmp

,token,tag,naive,aware,differs
0,The,DT,the,the,False
1,striped,JJ,striped,striped,False
2,bats,NNS,bat,bat,False
3,are,VBP,are,be,True
4,hanging,VBG,hanging,hang,True
5,on,IN,on,on,False
6,their,PRP$,their,their,False
7,feet,NNS,foot,foot,False
8,for,IN,for,for,False
9,best,JJS,best,best,False


**Reading the table:** only two words differ between naive and POS-aware lemmatization: `are` (naive `are` → aware **`be`**) and `hanging` (naive `hanging` → aware **`hang`**). Both are verb forms. The nouns (`bats→bat`, `feet→foot`, `results→result`) were already correct with the default noun assumption, and `striped` and `best` stay unchanged either way (`best` is not reduced to `good` by WordNet's lemmatizer here).

### Questions
1. **Difference from Exercise 1 / which words mattered most:** with POS information, inflected **verbs** (`running→run`, `was→be`, `are→be`, `hanging→hang`) and **adjectives** (`better→good`) are reduced correctly. Nouns were already right with the default. So the biggest gain is on irregular verbs and comparatives.
2. **Why grammatical information is needed:** the same surface form can have different lemmas depending on its part of speech. *better* as an adjective → `good`, as an adverb → `well`, as a verb ("to better oneself") → `better`. Without POS the lemmatizer can't choose, so it falls back on "noun" and typically returns the word unchanged.
3. **Codes:** `n` = noun, `v` = verb, `a` = adjective, `r` = adverb (WordNet also has `s` for satellite adjectives).

In [7]:
# Same word, three POS values
for pos in ["a", "r", "v", "n"]:
    print(pos, "->", lemmatizer.lemmatize("better", pos=pos))

a -> good
r -> well
v -> better
n -> better


## Exercise 3 — Ambiguous words and the limits of lemmatization

**3.1 Example sentences (two per word):**

In [8]:
examples = {
    "meeting": ["The meeting starts at 9.",              "She is meeting her advisor tomorrow."],
    "saw":     ["I saw a bird on the roof.",             "He cut the wood with a saw."],
    "flies":   ["Time flies when you are busy.",         "The flies landed on the fruit."],
    "better":  ["This is a better solution.",            "She sings better than me."],
    "left":    ["She left the room early.",              "Turn left at the corner."],
}
for w, sents in examples.items():
    for s in sents:
        print(f"{w:8s}| {s}")

meeting | The meeting starts at 9.
meeting | She is meeting her advisor tomorrow.
saw     | I saw a bird on the roof.
saw     | He cut the wood with a saw.
flies   | Time flies when you are busy.
flies   | The flies landed on the fruit.
better  | This is a better solution.
better  | She sings better than me.
left    | She left the room early.
left    | Turn left at the corner.


**3.2 – 3.3 POS tags and lemmas of the target word in every sentence:**

In [9]:
rows = []
for w, sents in examples.items():
    for s in sents:
        for tok, tag, lem in pos_aware_lemmatize(s):
            if tok.lower() == w:
                rows.append({"word": w, "sentence": s, "tag": tag,
                             "naive": lemmatizer.lemmatize(w), "pos-aware lemma": lem})
pd.DataFrame(rows)

,word,sentence,tag,naive,pos-aware lemma
0,meeting,The meeting starts at 9.,NN,meeting,meeting
1,meeting,She is meeting her advisor tomorrow.,VBG,meeting,meet
2,saw,I saw a bird on the roof.,VBD,saw,saw
3,saw,He cut the wood with a saw.,NN,saw,saw
4,flies,Time flies when you are busy.,NNS,fly,fly
5,flies,The flies landed on the fruit.,NNS,fly,fly
6,better,This is a better solution.,JJR,better,good
7,better,She sings better than me.,JJR,better,good
8,left,She left the room early.,VBD,left,leave
9,left,Turn left at the corner.,VBD,left,leave


**3.2 / 3.3 Analysis**

| word | did the tagger separate the two uses? | lemma result |
|---|---|---|
| **meeting** | **Yes** (`NN` vs `VBG`) | `meeting` (noun) vs **`meet`** (verb): different and correct |
| **saw** | **Yes** (`VBD` vs `NN`) | noun `saw` is correct, but the verb gives `saw` instead of **`see`**. The tagger is right here; the **lemmatizer** is the one failing, because WordNet also contains the verb *to saw* and returns that entry first |
| **flies** | **No** (`NNS` for both; "Time flies" should be a verb, `VBZ`) | `fly` in both cases, so no visible harm here (the lemma is the same), but the tag is wrong |
| **better** | Partly: `JJR` for both, but in "sings better" it should be an adverb (`RBR`) | adjective → `good` is right; for the adverb use the correct lemma is `well`, but the pipeline gives `good` |
| **left** | **No** for the second use: `VBD` for both, but "Turn left" is an adverb (or noun) | `leave` for both; correct for "She left the room", **wrong** for "Turn left" (should stay `left`) |

**3.4 Tagger mistakes that propagate:**
- "**Turn left** at the corner": the tagger labels *left* as a past-tense verb (`VBD`), so the lemmatizer is asked for the verb lemma and returns **`leave`**, which is a completely different word from the direction *left*. A search for "turn left" would then match "leave".
- "**She sings better** than me": *better* is tagged as an adjective comparative (`JJR`) rather than an adverb, so it is lemmatized to `good` instead of `well`.

In both cases the lemmatizer did exactly what it was told. The **wrong tag** produced the wrong lemma: errors in POS tagging flow straight into the lemmatization result.

### Question — why context matters
A word's lemma depends on its grammatical role, and the role can only be decided from the **surrounding context** (*saw* verb vs noun, *left* verb vs adverb). So lemmatization is **not independent** of POS tagging (Lab 8): the quality of the lemmas is capped by the quality of the tags, and a word-by-word lemmatizer with no context can only guess (it defaults to noun). In practice a lemmatizer should be treated as the second stage of a pipeline (tokenize → tag → lemmatize), and its errors include those inherited from the tagger.

### Challenge — derivational vs inflectional morphology

In [10]:
print(lemmatizer.lemmatize("organization", pos="n"))
print(lemmatizer.lemmatize("happiness", pos="n"))

organization
happiness


Both come back unchanged: **`organization`** and **`happiness`**.

WordNet's lemmatizer implements **inflectional** morphology only: it strips endings that mark plural, tense or comparison (`-s`, `-ed`, `-ing`, `-er`, `-est`) and always checks the result against the WordNet dictionary, returning a form only if it exists there. *organization* → *organize* and *happiness* → *happy* are **derivational** changes: they create a new word with a new part of speech and a (slightly) different meaning, so they are different **lemmas**, and each of them has its own WordNet entry. A lemma is a dictionary headword; *organization* and *happiness* already are headwords, so there is nothing to reduce.

A stemmer, by contrast, blindly chops suffixes (`organization → organ`, `happiness → happi`) without a dictionary check. That gives more aggressive conflation (and more errors, like *organ*) but doesn't need POS information or a lexicon.